# Exercise 02 — Typed Models

When you're building with LLMs, you'll constantly be moving structured data around: documents going into RAG, search results coming back, API responses getting parsed. If that data has no types, bugs hide until runtime and your editor can't help you.

This exercise covers two approaches: plain `dataclasses` from the standard library, and `Pydantic` models. You'll use both in real AI projects, so it's worth knowing the difference.

**What you'll practice:**
- Defining typed classes with `@dataclass`
- Validating data with Pydantic `BaseModel`
- Understanding when each one makes sense

In [ ]:
!pip install pydantic -q

## Part 1 — Dataclasses

A `dataclass` is just a class where Python writes the boring `__init__` for you. No validation, no coercion. Fast and lightweight.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime


@dataclass
class Document:
    id: str
    content: str
    source: str
    created_at: datetime = field(default_factory=datetime.now)
    tags: list[str] = field(default_factory=list)


doc = Document(
    id="doc-001",
    content="LangGraph is a library for building stateful agents as graphs.",
    source="langchain-docs",
    tags=["agents", "langgraph"],
)

print(doc)
print(f"Source: {doc.source}")
print(f"Tags: {doc.tags}")

## Part 2 — Pydantic models

Pydantic validates the data when you create the object. If the types are wrong, it tells you immediately with a clear error. This is what you'll use for structured LLM output, API responses, and anything coming from outside your code.

In [ ]:
from pydantic import BaseModel, Field


class SearchResult(BaseModel):
    chunk_id: str
    content: str
    score: float = Field(ge=0.0, le=1.0, description="Similarity score between 0 and 1")
    source_document: str


result = SearchResult(
    chunk_id="chunk-042",
    content="RAG combines retrieval with generation to ground responses in real data.",
    score=0.91,
    source_document="doc-001",
)

print(result.model_dump())
print(f"Score: {result.score}")

In [ ]:
# See what happens when validation fails
# Pydantic tells you exactly what's wrong and where

try:
    bad_result = SearchResult(
        chunk_id="chunk-043",
        content="This one has a bad score.",
        score=1.5,  # out of range
        source_document="doc-001",
    )
except Exception as e:
    print(f"Validation error:\n{e}")

## Check your understanding

- `dataclass` vs Pydantic: when would you pick one over the other?
- What does `field(default_factory=list)` do, and why is `default=[]` dangerous in a dataclass?
- Add a `metadata: dict` field to `Document` and populate it with some key-value pairs.